# Llama3 with Bio Cards & Context Manager

Fast scene-batch validation using Llama3 with speaker bio cards and context manager.
Sends full dialogue scenes to Llama with enriched context, then writes predictions to CSV.

In [45]:
import os
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd
import sys
sys.path.append("../")

from src.llama_sft_function_calls import call_llama3_context_manager, llama3_sft_call
from src.load_data import load_data_from_csv
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm

In [46]:
# --- CONFIGURATION ---
BASE_DIR = "../"
DATA_PATH = os.path.join(BASE_DIR, "data", "test_sent_emo.csv")
BIO_CARDS_PATH = os.path.join(BASE_DIR, "logs", "speaker_bio_cards.json")
OUTPUT_DIR = os.path.join(BASE_DIR, "logs", "llama3")

os.makedirs(OUTPUT_DIR, exist_ok=True)

LLAMA3_PROMPT = """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert Emotion Recognition assistant specialized in the MELD benchmark.
You have access to speaker biographical information and scene context.
For each scene, predict ONE emotion for EVERY utterance using labels: [anger, disgust, fear, joy, neutral, sadness, surprise].
Consider speaker personalities, relationships, and emotional context when making predictions.
Return JSON only in this schema:
{
  "predictions": [
    {"utterance_id": "id", "predicted_emotion": "label", "reasoning": "short reason"}
  ]
}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

print(f"Configuration:")
print(f"  Data: {DATA_PATH}")
print(f"  Bio Cards: {BIO_CARDS_PATH}")
print(f"  Output: {OUTPUT_DIR}")

Configuration:
  Data: ../data\test_sent_emo.csv
  Bio Cards: ../logs\speaker_bio_cards.json
  Output: ../logs\llama3


In [47]:
# --- UTILS ---
def load_speaker_bio_cards(json_file=BIO_CARDS_PATH):
    """Load speaker biographical information."""
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"Bio cards error: {e}")
        return {}

def format_bio_cards_context(bio_cards_dict, speakers_in_scene):
    """Format bio cards for the speakers in this scene."""
    if not bio_cards_dict:
        return ""
    
    context = "\n--- SPEAKER PROFILES ---\n"
    for speaker in speakers_in_scene:
        if speaker in bio_cards_dict:
            bio = bio_cards_dict[speaker]
            if isinstance(bio, dict):
                bio_text = "\n".join([f"{k}: {v}" for k, v in bio.items()])
            else:
                bio_text = str(bio)
            context += f"\n{speaker}:\n{bio_text}\n"
    return context

def parse_scene_predictions(text: str) -> dict:
    """Extract predictions from Llama response."""
    if not text:
        return {}
    
    try:
        match = re.search(r"\{[\s\S]*\}", text)
        if match:
            parsed = json.loads(match.group(0))
    except Exception:
        return {}
    
    if parsed is None:
        return {}
    
    predictions = []
    if isinstance(parsed, dict):
        maybe_list = parsed.get("predictions", [])
        if isinstance(maybe_list, list):
            predictions = maybe_list
    elif isinstance(parsed, list):
        predictions = parsed
    
    out = {}
    for item in predictions:
        if not isinstance(item, dict):
            continue
        utt_id = item.get("utterance_id")
        if utt_id is None:
            continue
        emotion = item.get("predicted_emotion") or item.get("emotion")
        reason = item.get("reasoning", "")
        out[str(utt_id)] = {
            "predicted_emotion": emotion.lower().strip() if isinstance(emotion, str) else None,
            "reasoning": str(reason) if reason is not None else ""
        }
    return out

print("Utility functions loaded.")

Utility functions loaded.


In [48]:
# --- LOAD DATA AND BIO CARDS ---
print("Loading data...")
df = load_data_from_csv(DATA_PATH)
df = df.sort_values(["Dialogue_ID", "Utterance_ID"]).reset_index(drop=True)

print("Loading bio cards...")
bio_cards = load_speaker_bio_cards()
print(f"Loaded {len(bio_cards)} speaker profiles")

print(f"Total data rows: {len(df)}")
print(f"Unique dialogues: {df['Dialogue_ID'].nunique()}")

Loading data...
Data loaded successfully from ../data\test_sent_emo.csv
Loading bio cards...
Loaded 260 speaker profiles
Total data rows: 2610
Unique dialogues: 280


In [49]:
def build_scene_batches_with_context(df: pd.DataFrame, bio_cards: dict) -> list:
    """
    Build scenes with context manager + bio cards.
    Each scene includes enriched context.
    """
    scenes = []
    
    for dialogue_id, group in df.groupby("Dialogue_ID", sort=False):
        scene_rows = []
        scene_lines = []
        speakers_in_scene = set()
        
        for _, row in group.iterrows():
            utterance_id = str(row.get("Utterance_ID", "NA"))
            speaker = str(row.get("Speaker", "Unknown"))
            utterance = str(row.get("Utterance", ""))
            gt_emotion = str(row.get("Emotion", "")).strip().lower()
            gt_sentiment = str(row.get("Sentiment", "")).strip().lower()
            
            speakers_in_scene.add(speaker)
            
            scene_rows.append({
                "Dialogue_ID": dialogue_id,
                "Utterance_ID": utterance_id,
                "Recognition_ID": f"{dialogue_id}_{utterance_id}",
                "Speaker": speaker,
                "Utterance": utterance,
                "ground_truth_emotion": gt_emotion,
                "ground_truth_sentiment": gt_sentiment,
            })
            scene_lines.append(f"{utterance_id} | {speaker}: {utterance}")
        
        # Build enriched prompt with bio cards
        bio_context = format_bio_cards_context(bio_cards, speakers_in_scene)
        
        scene_dialogue = "\n".join(scene_lines)
        
        scene_prompt = (
            f"{LLAMA3_PROMPT}\n"
            f"Scene Dialogue ID: {dialogue_id}\n"
            f"{bio_context}\n"
            f"--- SCENE DIALOGUE ---\n"
            f"Utterances in order:\n"
            f"{scene_dialogue}\n\n"
            f"Return predictions for ALL utterance_id values above."
        )
        
        scenes.append({
            "dialogue_id": dialogue_id,
            "rows": scene_rows,
            "prompt": scene_prompt,
            "speakers": list(speakers_in_scene)
        })
    
    return scenes

print("Building scenes with context...")
scenes = build_scene_batches_with_context(df, bio_cards)
print(f"Prepared {len(scenes)} scenes")

Building scenes with context...
Prepared 280 scenes


In [50]:
def run_inference(scenes: list, limit: Optional[int] = None):
    """
    Run Llama3 inference on scenes with bio cards + context manager analysis.
    Saves results incrementally after each scene (safe for interruption).
    Retries each scene once on error before skipping.
    """
    if limit is not None:
        scenes = scenes[:limit]
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(OUTPUT_DIR, f"llama3_biocards_context_predictions_{timestamp}.csv")
    
    records = []
    total_scenes = len(scenes)
    total_utterances = sum(len(s["rows"]) for s in scenes)
    
    print(f"\nRunning inference on {total_scenes} scenes ({total_utterances} utterances)...\n")
    print(f"Saving incrementally to: {output_path}\n")
    
    system_instruction = """You are an expert Emotion Recognition assistant specialized in the MELD benchmark.
You have access to speaker biographical information and scene context.
For each scene, predict ONE emotion for EVERY utterance using labels: [anger, disgust, fear, joy, neutral, sadness, surprise].
Consider speaker personalities, relationships, and emotional context when making predictions.
Return JSON only in this schema:
{
  "predictions": [
    {"utterance_id": "id", "predicted_emotion": "label", "reasoning": "short reason"}
  ]
}"""
    
    for scene_idx, scene in enumerate(tqdm(scenes, desc="Scenes", ncols=80), start=1):
        dialogue_id = scene["dialogue_id"]
        max_retries = 2  # Initial attempt + 1 retry
        
        for attempt in range(max_retries):
            try:
                # Step 1: Get context manager analysis for the scene
                scene_dialogue = "\n".join([f"{row['Utterance_ID']} | {row['Speaker']}: {row['Utterance']}" 
                                           for row in scene["rows"]])
                
                context_analysis = call_llama3_context_manager(scene_script=scene_dialogue)
                
                # Step 2: Build enriched user content with bio cards + context analysis
                bio_context = format_bio_cards_context(bio_cards, scene["speakers"])
                
                user_content = (
                    f"Scene Dialogue ID: {dialogue_id}\n"
                    f"{bio_context}\n"
                    f"--- SCENE CONTEXT ANALYSIS ---\n"
                    f"{context_analysis}\n\n"
                    f"--- SCENE DIALOGUE ---\n"
                    f"Utterances in order:\n"
                    f"{scene_dialogue}\n\n"
                    f"Using the speaker profiles, context analysis, and dialogue above, "
                    f"return predictions for ALL utterance_id values."
                )
                
                # Step 3: Get emotion predictions
                model_output = llama3_sft_call(
                    system_instruction=system_instruction,
                    user_content=user_content
                )
                
                pred_map = parse_scene_predictions(model_output)
                
                for row in scene["rows"]:
                    pred_item = pred_map.get(str(row["Utterance_ID"]), {})
                    records.append({
                        "Dialogue_ID": row["Dialogue_ID"],
                        "Utterance_ID": row["Utterance_ID"],
                        "Recognition_ID": row["Recognition_ID"],
                        "Speaker": row["Speaker"],
                        "Utterance": row["Utterance"],
                        "ground_truth_emotion": row["ground_truth_emotion"],
                        "ground_truth_sentiment": row["ground_truth_sentiment"],
                        "predicted_emotion": pred_item.get("predicted_emotion"),
                        "reasoning": pred_item.get("reasoning", ""),
                        "scene_raw_output": model_output,
                        "context_analysis": context_analysis
                    })
                
                # Save incrementally after each scene (safe for interruption)
                results_df = pd.DataFrame(records)
                results_df.to_csv(output_path, index=False)
                
                # Success - break out of retry loop
                break
                
            except Exception as e:
                error_msg = str(e)[:100]
                if attempt < max_retries - 1:
                    print(f"  ⚠️  Error in scene {dialogue_id} (attempt {attempt + 1}): {error_msg}")
                    print(f"      Retrying...")
                else:
                    print(f"  ❌ Error in scene {dialogue_id} (skipped after {max_retries} attempts): {error_msg}")
                continue
    
    # Final save
    results_df = pd.DataFrame(records)
    results_df.to_csv(output_path, index=False)
    
    print(f"\n✅ Saved {len(records)} predictions to {output_path}")
    
    # Evaluate
    valid_df = results_df.dropna(subset=["ground_truth_emotion", "predicted_emotion"]).copy()
    if not valid_df.empty:
        labels = sorted(valid_df["ground_truth_emotion"].unique().tolist())
        print(f"\n📊 Evaluation ({len(valid_df)} valid predictions):")
        print(classification_report(
            valid_df["ground_truth_emotion"],
            valid_df["predicted_emotion"],
            labels=labels,
            digits=4,
            zero_division=0
        ))
        wf1 = f1_score(
            valid_df["ground_truth_emotion"],
            valid_df["predicted_emotion"],
            average="weighted",
            zero_division=0
        )
        print(f"Weighted F1: {wf1:.4f}")
    
    return output_path, results_df

print("Inference function ready.")

Inference function ready.


In [51]:
# --- RUN INFERENCE ---
# Set limit=None to run on all scenes, or limit=10 for a quick test
output_csv, results_df = run_inference(scenes, limit=10)

print(f"\nResults saved to: {output_csv}")
print(f"Preview:")
print(results_df.head(10))


Running inference on 10 scenes (84 utterances)...

Saving incrementally to: ../logs\llama3\llama3_biocards_context_predictions_20260425_172335.csv



Scenes:  10%|███▌                                | 1/10 [00:14<02:10, 14.46s/it]


KeyboardInterrupt: 

In [ ]:
# --- MERGE ALL TIMESTAMPED PREDICTIONS INTO MASTER FILE ---
import glob

print("Looking for all prediction files...")
pred_files = glob.glob(os.path.join(OUTPUT_DIR, "llama3_biocards_context_predictions_*.csv"))
pred_files = [f for f in pred_files if "combined" not in f and "master" not in f]  # Exclude combined/master files

if pred_files:
    print(f"Found {len(pred_files)} timestamped prediction files:")
    for f in sorted(pred_files):
        print(f"  - {os.path.basename(f)}")
    
    # Load and merge all files
    all_dfs = []
    total_rows = 0
    for pred_file in pred_files:
        df_temp = pd.read_csv(pred_file)
        all_dfs.append(df_temp)
        total_rows += len(df_temp)
        print(f"    Loaded {len(df_temp)} rows from {os.path.basename(pred_file)}")
    
    # Merge and deduplicate by Recognition_ID (in case of overlaps)
    merged_df = pd.concat(all_dfs, ignore_index=True)
    merged_df = merged_df.drop_duplicates(subset=['Recognition_ID'], keep='first')
    
    print(f"\nMerged {total_rows} total rows into {len(merged_df)} unique predictions")
    
    # Save as master file (no timestamp)
    master_path = os.path.join(OUTPUT_DIR, "llama3_biocards_context_predictions_master.csv")
    merged_df.to_csv(master_path, index=False)
    print(f"✅ Saved master file: {os.path.basename(master_path)}")
    
    # Update results_df for next steps
    results_df = merged_df
    
else:
    print("❌ No timestamped prediction files found!")
    results_df = pd.DataFrame()

print(f"\n{'='*80}")
print(f"Master file ready with {len(results_df)} predictions")
print(f"Proceeding to process remaining scenes...")
print(f"{'='*80}\n")

Loaded test results: 84 predictions
Total scenes available: 280
Already processed: 10 dialogues
Remaining scenes to process: 270

Running inference on remaining scenes...


Running inference on 270 scenes (2526 utterances)...

Saving incrementally to: ../logs\llama3\llama3_biocards_context_predictions_20260425_141605.csv



Scenes:  12%|███▉                            | 33/270 [18:18<2:09:23, 32.76s/it]

  ⚠️  Error in scene 43 (attempt 1): 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/g
      Retrying...


Scenes:  16%|████▉                           | 42/270 [40:33<3:35:21, 56.67s/it]

  ⚠️  Error in scene 52 (attempt 1): 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/g
      Retrying...


...

  ⚠️  Error in scene 134 (attempt 1): 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/g
      Retrying...


Scenes:  46%|████████████▌              | 125/270 [2:30:20<15:35:03, 386.92s/it]

  ❌ Error in scene 134 (skipped after 2 attempts): 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/g
  ⚠️  Error in scene 135 (attempt 1): 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/g
      Retrying...


Scenes:  47%|█████████████▋               | 128/270 [2:48:25<3:06:51, 78.95s/it]


KeyboardInterrupt: 

In [ ]:
# --- CONTINUE INFERENCE FROM MASTER FILE ---
# Uses the merged master file to track progress and continue on remaining scenes

print(f"Starting with master file: {len(results_df)} predictions already saved")
print(f"Total scenes available: {len(scenes)}")

# Get dialogue_ids already processed
processed_dialogue_ids = set(results_df['Dialogue_ID'].unique())
print(f"Already processed: {len(processed_dialogue_ids)} dialogues")

# Calculate stats on what's already done
already_done_utterances = len(results_df)
total_utterances = sum(len(s["rows"]) for s in scenes)
print(f"Already processed: {already_done_utterances}/{total_utterances} utterances ({already_done_utterances*100//total_utterances}%)")

# Filter to only remaining scenes
remaining_scenes = [s for s in scenes if s['dialogue_id'] not in processed_dialogue_ids]
print(f"Remaining scenes to process: {len(remaining_scenes)}")
remaining_utterances = sum(len(s["rows"]) for s in remaining_scenes)
print(f"Remaining utterances: {remaining_utterances}")

if remaining_scenes:
    # Run inference on remaining scenes
    print("\n" + "="*80)
    print("Running inference on remaining scenes...")
    print("="*80 + "\n")
    
    remaining_output_csv, remaining_results_df = run_inference(remaining_scenes, limit=None)
    
    # Combine results
    combined_df = pd.concat([results_df, remaining_results_df], ignore_index=True)
    
    # Save as updated master file
    master_path = os.path.join(OUTPUT_DIR, "llama3_biocards_context_predictions_master.csv")
    combined_df.to_csv(master_path, index=False)
    
    print(f"\n✅ Updated master file saved: {os.path.basename(master_path)}")
    print(f"   Total predictions now: {len(combined_df)}")
    print(f"   Progress: {len(combined_df)}/{total_utterances} utterances ({len(combined_df)*100//total_utterances}%)")
    
    print(f"\nPreview of combined results:")
    print(combined_df.head(10))
    
    results_df = combined_df
else:
    print("✅ All scenes already processed!")

✅ Loaded 1201 predictions from:
   llama3_biocards_context_predictions_20260425_141605.csv

Proceeding to continue inference on remaining scenes...


In [ ]:
# --- CLASSIFICATION REPORT FOR PREDICTIONS ---
# Automatically uses the master file for evaluation

master_file = os.path.join(OUTPUT_DIR, "llama3_biocards_context_predictions_master.csv")

if not os.path.exists(master_file):
    print(f"❌ Master file not found: {master_file}")
    print("Please run cell 8 first to create/merge prediction files")
else:
    pred_df = pd.read_csv(master_file)
    print(f"✅ Loaded master file: {len(pred_df)} predictions")
    print(f"\nColumns: {list(pred_df.columns)}\n")
    
    # Filter valid predictions
    valid_df = pred_df.dropna(subset=["ground_truth_emotion", "predicted_emotion"]).copy()
    print(f"Valid predictions (both fields present): {len(valid_df)}\n")
    
    if len(valid_df) > 0:
        # Get all unique emotion labels
        all_emotions = sorted(set(
            list(valid_df["ground_truth_emotion"].unique()) + 
            list(valid_df["predicted_emotion"].unique())
        ))
        
        print(f"Emotion classes: {all_emotions}\n")
        print("="*80)
        print("CLASSIFICATION REPORT")
        print("="*80 + "\n")
        
        # Generate detailed report
        print(classification_report(
            valid_df["ground_truth_emotion"],
            valid_df["predicted_emotion"],
            labels=all_emotions,
            digits=4,
            zero_division=0
        ))
        
        # Weighted F1 score
        wf1 = f1_score(
            valid_df["ground_truth_emotion"],
            valid_df["predicted_emotion"],
            average="weighted",
            zero_division=0
        )
        
        # Macro F1 score
        mf1 = f1_score(
            valid_df["ground_truth_emotion"],
            valid_df["predicted_emotion"],
            average="macro",
            zero_division=0
        )
        
        # Accuracy
        accuracy = (valid_df["ground_truth_emotion"] == valid_df["predicted_emotion"]).sum() / len(valid_df)
        
        print("\n" + "="*80)
        print("SUMMARY METRICS")
        print("="*80)
        print(f"Overall Accuracy:    {accuracy:.4f}")
        print(f"Weighted F1-Score:   {wf1:.4f}")
        print(f"Macro F1-Score:      {mf1:.4f}")
        print(f"Total Predictions:   {len(valid_df)}")
        print("="*80)
        
        # Show confusion matrix
        from sklearn.metrics import confusion_matrix
        import numpy as np
        
        cm = confusion_matrix(
            valid_df["ground_truth_emotion"],
            valid_df["predicted_emotion"],
            labels=all_emotions
        )
        
        print("\nConfusion Matrix (rows=ground_truth, cols=predicted):")
        print(f"Emotions: {all_emotions}")
        print(cm)
        
    else:
        print("❌ No valid predictions found!")

Loaded 1201 predictions from file

Columns: ['Dialogue_ID', 'Utterance_ID', 'Recognition_ID', 'Speaker', 'Utterance', 'ground_truth_emotion', 'ground_truth_sentiment', 'predicted_emotion', 'reasoning', 'scene_raw_output', 'context_analysis']

Valid predictions (both fields present): 1183
Emotion classes: ['anger', 'desire', 'despair', 'disgust', 'fear', 'joy', 'negative', 'negative surprise', 'neutral', 'positive', 'positive surprise', 'sadness', 'surprise']

CLASSIFICATION REPORT

                   precision    recall  f1-score   support

            anger     0.8000    0.6410    0.7117       156
           desire     0.0000    0.0000    0.0000         0
          despair     0.0000    0.0000    0.0000         0
          disgust     0.8182    0.5294    0.6429        34
             fear     0.6296    0.6296    0.6296        27
              joy     0.6828    0.6580    0.6702       193
         negative     0.0000    0.0000    0.0000         0
negative surprise     0.0000    0.0000  